# Model Comparison

사출성형 품질 분석 팀 프로젝트의 기존 과제 노트북입니다.
팀: 락스메이, 정유진, 모티타.

**게시용 정리:** 원 분석 코드는 유지하고 파일명·데이터 경로를 정리했습니다.
이 노트북의 표와 출력은 기존 실행 기록이며, 게시 시점에 다시 실행한 결과가 아닙니다.
대표 그림은 `results/figures/`로 분리했습니다. 원본에 없던 실험 결과를 추가하지 않았습니다.

**라벨:** 최종 모델에서 `1=정상(Y)`, `0=불량(N)`입니다. 기본 precision/recall/F1은 정상 클래스 기준입니다.
문서에 남아 있는 defect/positive 설명은 이 라벨 정의와 함께 읽어야 합니다.
성능 해석과 전처리 한계는 [실험 기록](../docs/experiment-notes.md)을 확인하세요.

저장소 루트에서 `jupyter lab`을 실행하고 이 파일을 열어 순서대로 실행하세요.


In [ ]:
from pathlib import Path
Path("../outputs").mkdir(exist_ok=True)


# KAMP Injection Molding – End-to-End Data Science Project

**Course**: Data & EDA Foundations <br>
**Project team**: 락스메이, 정유진, 모티타  
**Dataset**: KAMP Injection Molding (labeled_data.csv)  
**Goal**: Build an end-to-end data science pipeline to understand and predict product quality
(Pass/Fail) in an injection molding manufacturing process.

We follow a CRISP-DM workflow:

1. Business Understanding  
2. Data Understanding & Exploratory Data Analysis (EDA)  
3. Data Quality & Preparation (missing values, outliers, transformations, encoding)  
4. Modeling & Evaluation (baseline classifier for Pass/Fail)  
5. Conclusions & Recommendations


## Importing libraries


The libraries we'll import to start with are pandas and numpy for data handling, and matplotlib, seaborn, and plotly for display generation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
sns.set_style("darkgrid")


## 1. Business Understanding

### 1.1 Background & Process (Injection Molding)
- **Injection Molding Overview**:Injection molding is a high-throughput thermoplastic manufacturing process consisting of three main phases— **filling, pressurization**, and **compensation/holding**. The polymer granules are dried and fed into the hopper, melted in the barrel, injected into the mold cavity under controlled temperature and pressure, and finally ejected after cooling. This cycle repeats rapidly in production  
- **Typical Defects**:Common quality issues include short-shot, flash, sink marks, flow marks, weld lines, bubbles, and warpage, often arising from inappropriate process parameter combinations.
- **Key Controllable Factors**:Major process parameters influencing product quality include temperatures (barrel, hopper, mold), pressures (injection, transfer, back pressure), speeds (injection speed, screw RPM), and timings (filling, holding, cooling, cycle time)
- **Data Context**:The dataset originates from an automotive windshield side molding injection line, collected at 0.2-second intervals between October 16 and November 19, 2020, under the KAMP initiative (Ministry of SMEs and Startups & KAIST)

### 1.2 Problem Definition
- **Objective**:Predict or monitor **Pass/Fail** (good vs defective) outcomes using process sensor data, enabling early detection of anomalies and identification of root causes for quality drift.
-**Field Pain Points**:
 1. Defects are typically detected only after cooling, delaying corrective action.
 2. Complex, interacting parameters make root-cause diagnosis difficult. 3.Parameter tuning remains highly experience-dependent.
-**Scope**:The initial focus is on early warning and cause identification through real-time or batch monitoring. Long-term, insights may inform recipe optimization for process parameters (e.g., speed, temperature, pressure).

### 1.3 Stakeholders
- **Operators** & **process engineers** - Real time parameter tuning and process stability.
- **Quality teams** - Defect inspection, classification, and triage.
- **Managers** - Scrap/cost control, throughput, and KPI tracking.

### 1.4 Analysis Purpose
- Build foundational **data understanding**: schema, types, ranges, missingness, simple correlations.
- Identify early **data issues**: duplicates, anomalies (e.g., zeros where impossible), potential leakage fields.
- Produce **actionable insights** to guide the next modeling phase.


## Loading and exploring the data

In the CRISP-DM framework, **Data Understanding** starts right after we clarify the business problem.  
In this section we load a real dataset from a plastic injection molding process. Each row represents one shot, with several process variables (temperatures, times, positions, volumes, etc.) and a final **quality** label.

We first look at the basic structure of the dataset (`df.info()` and `df.head()`) to answer questions such as:

- How many samples do we have?
- How many variables and of what type?
- Are there missing values or strange data types?

This quick scan gives us a first feeling for data quality and helps us plan the next steps in data preparation and modeling.


In [ ]:
from sklearn.preprocessing import StandardScaler, PowerTransformer, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline

from sklearn.model_selection import train_test_split

from sklearn.metrics import f1_score, confusion_matrix, classification_report

In [ ]:
df = pd.read_csv('../data/labeled_data.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7996 entries, 0 to 7995
Data columns (total 45 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   _id                       7996 non-null   object 
 1   TimeStamp                 7996 non-null   object 
 2   PART_FACT_PLAN_DATE       7996 non-null   object 
 3   PART_FACT_SERIAL          7996 non-null   int64  
 4   PART_NAME                 7996 non-null   object 
 5   EQUIP_CD                  7996 non-null   object 
 6   EQUIP_NAME                7996 non-null   object 
 7   PassOrFail                7996 non-null   object 
 8   Reason                    71 non-null     object 
 9   Injection_Time            7996 non-null   float64
 10  Filling_Time              7996 non-null   float64
 11  Plasticizing_Time         7996 non-null   float64
 12  Cycle_Time                7996 non-null   float64
 13  Clamp_Close_Time          7996 non-null   float64
 14  Cushion_

In [ ]:
df.describe()

       PART_FACT_SERIAL  Injection_Time  Filling_Time  Plasticizing_Time  \
count       7996.000000     7996.000000   7996.000000        7996.000000   
mean          11.248749        8.242189      3.896309          16.209505   
std            5.005191        3.107724      1.284451           1.443720   
min            2.000000        1.050000      0.930000          10.920000   
25%            9.000000        9.520000      4.400000          16.580000   
50%           10.000000        9.560000      4.440000          16.799999   
75%           13.000000        9.590000      4.470000          16.889999   
max           24.000000       16.309999      8.270000          21.120001   

        Cycle_Time  Clamp_Close_Time  Cushion_Position  Switch_Over_Position  \
count  7996.000000       7996.000000       7996.000000           7996.000000   
mean     59.900902          7.064088        653.408195              0.330320   
std       0.885680          0.136595         10.165106             14.65488

In [ ]:
df.head()

                        _id            TimeStamp     PART_FACT_PLAN_DATE  \
0  5f8928bb9c0189cc666ef19b  2020-10-16 04:57:47  2020-10-16 오전 12:00:00   
1  5f8928de9c0189cc666ef20b  2020-10-16 04:58:48  2020-10-16 오전 12:00:00   
2  5f8928df9c0189cc666ef213  2020-10-16 04:58:48  2020-10-16 오전 12:00:00   
3  5f8928f39c0189cc666ef25e  2020-10-16 04:59:48  2020-10-16 오전 12:00:00   
4  5f8928f59c0189cc666ef265  2020-10-16 04:59:48  2020-10-16 오전 12:00:00   

   PART_FACT_SERIAL              PART_NAME EQUIP_CD  EQUIP_NAME PassOrFail  \
0                24  CN7 W/S SIDE MLD'G RH      S14  650톤-우진2호기          Y   
1                24  CN7 W/S SIDE MLD'G RH      S14  650톤-우진2호기          Y   
2                23  CN7 W/S SIDE MLD'G LH      S14  650톤-우진2호기          Y   
3                23  CN7 W/S SIDE MLD'G LH      S14  650톤-우진2호기          Y   
4                24  CN7 W/S SIDE MLD'G RH      S14  650톤-우진2호기          Y   

  Reason  Injection_Time  ...  Mold_Temperature_3  Mold_Temperature_4  \
0

In [ ]:
desc = df.describe().T
display(desc)

                           count        mean         std         min  \
PART_FACT_SERIAL          7996.0   11.248749    5.005191    2.000000   
Injection_Time            7996.0    8.242189    3.107724    1.050000   
Filling_Time              7996.0    3.896309    1.284451    0.930000   
Plasticizing_Time         7996.0   16.209505    1.443720   10.920000   
Cycle_Time                7996.0   59.900902    0.885680   58.840000   
Clamp_Close_Time          7996.0    7.064088    0.136595    5.160000   
Cushion_Position          7996.0  653.408195   10.165106   11.100000   
Switch_Over_Position      7996.0    0.330320   14.654889    0.000000   
Plasticizing_Position     7996.0   66.054484    5.418233   53.549999   
Clamp_Open_Position       7996.0  544.244111  236.351404    4.630000   
Max_Injection_Speed       7996.0   66.923612   26.460186   22.299999   
Max_Screw_RPM             7996.0   30.701989    0.159179   27.100000   
Average_Screw_RPM         7996.0  149.023424  130.828781   25.50

## Exploring target variable

Before modeling, we need to understand the **target variable**. Here, the original column `PassOrFail` encodes whether a part passes or fails quality inspection.

We start by plotting the distribution of `PassorFail`.
We create a new binary variable `class`, where:
- `class = 1` pass = 1,  
- `class = 0` fail = 0.

Working with a clean, well-defined target variable is part of **data preparation** and will make our classification task clearer.


In [ ]:
df['class'] = df['PassOrFail'].apply(lambda x: 1 if x == 'Y' else 0)
df.head(3)

                        _id            TimeStamp     PART_FACT_PLAN_DATE  \
0  5f8928bb9c0189cc666ef19b  2020-10-16 04:57:47  2020-10-16 오전 12:00:00   
1  5f8928de9c0189cc666ef20b  2020-10-16 04:58:48  2020-10-16 오전 12:00:00   
2  5f8928df9c0189cc666ef213  2020-10-16 04:58:48  2020-10-16 오전 12:00:00   

   PART_FACT_SERIAL              PART_NAME EQUIP_CD  EQUIP_NAME PassOrFail  \
0                24  CN7 W/S SIDE MLD'G RH      S14  650톤-우진2호기          Y   
1                24  CN7 W/S SIDE MLD'G RH      S14  650톤-우진2호기          Y   
2                23  CN7 W/S SIDE MLD'G LH      S14  650톤-우진2호기          Y   

  Reason  Injection_Time  ...  Mold_Temperature_4  Mold_Temperature_5  \
0    NaN            9.59  ...                27.5                 0.0   
1    NaN            9.60  ...                27.6                 0.0   
2    NaN            9.60  ...                27.6                 0.0   

   Mold_Temperature_6  Mold_Temperature_7  Mold_Temperature_8  \
0                 0.0   

In [ ]:
print("Pass / Fail Counts:")
display(df["class"].value_counts())

Pass / Fail Counts:


class
1    7925
0      71
Name: count, dtype: int64

In [ ]:
sns.countplot(df, x='class', hue='class')

<Axes: xlabel='class', ylabel='count'>

[Historical plot output omitted; rerun this cell to display it.]

In [ ]:
df = df.drop('PassOrFail', axis=1)
df.head(3)

                        _id            TimeStamp     PART_FACT_PLAN_DATE  \
0  5f8928bb9c0189cc666ef19b  2020-10-16 04:57:47  2020-10-16 오전 12:00:00   
1  5f8928de9c0189cc666ef20b  2020-10-16 04:58:48  2020-10-16 오전 12:00:00   
2  5f8928df9c0189cc666ef213  2020-10-16 04:58:48  2020-10-16 오전 12:00:00   

   PART_FACT_SERIAL              PART_NAME EQUIP_CD  EQUIP_NAME Reason  \
0                24  CN7 W/S SIDE MLD'G RH      S14  650톤-우진2호기    NaN   
1                24  CN7 W/S SIDE MLD'G RH      S14  650톤-우진2호기    NaN   
2                23  CN7 W/S SIDE MLD'G LH      S14  650톤-우진2호기    NaN   

   Injection_Time  Filling_Time  ...  Mold_Temperature_4  Mold_Temperature_5  \
0            9.59          4.47  ...                27.5                 0.0   
1            9.60          4.48  ...                27.6                 0.0   
2            9.60          4.48  ...                27.6                 0.0   

   Mold_Temperature_6  Mold_Temperature_7  Mold_Temperature_8  \
0           

## Exploring predictors

Next we explore the **predictor variables** (features). The KDE plots show the distribution of each process variable, helping us see whether values are concentrated in a narrow range or spread out, and whether there are obvious outliers.

We also compute a **correlation matrix** to identify pairs of variables that are strongly positively or negatively correlated (in this exercise, greater than 0.5). Highly correlated variables may contain redundant information, but they can also hint at important process relationships (for example, between shot volume and screw position).

Finally, the scatter plots with `hue='class'` let us visually inspect how the target class is distributed in the feature space. If we can see some separation between good and defective parts in 2D plots, that suggests that a machine learning model may be able to learn useful decision boundaries.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

num_df = df.select_dtypes(include=[np.number]).copy()

drop_cols = ["PART_FACT_SERIAL", "class", "PassOrFail_bin"]
for c in drop_cols:
    if c in num_df.columns:
        num_df = num_df.drop(columns=c)

bad_cols = [c for c in num_df.columns
            if (num_df[c].nunique() <= 1) or (num_df[c].std() == 0)]

print("Dropping constant columns (cannot do KDE):", bad_cols)
num_df = num_df.drop(columns=bad_cols)

print("Final columns used for KDE:")
print(num_df.columns.tolist())

axarr = num_df.plot(kind="kde",
                    subplots=True,
                    figsize=(6, 3 * len(num_df.columns)),
                    sharex=False)
plt.tight_layout()
plt.show()



Dropping constant columns (cannot do KDE): ['Mold_Temperature_1', 'Mold_Temperature_2', 'Mold_Temperature_5', 'Mold_Temperature_6', 'Mold_Temperature_7', 'Mold_Temperature_8', 'Mold_Temperature_9', 'Mold_Temperature_10', 'Mold_Temperature_11', 'Mold_Temperature_12']
Final columns used for KDE:
['Injection_Time', 'Filling_Time', 'Plasticizing_Time', 'Cycle_Time', 'Clamp_Close_Time', 'Cushion_Position', 'Switch_Over_Position', 'Plasticizing_Position', 'Clamp_Open_Position', 'Max_Injection_Speed', 'Max_Screw_RPM', 'Average_Screw_RPM', 'Max_Injection_Pressure', 'Max_Switch_Over_Pressure', 'Max_Back_Pressure', 'Average_Back_Pressure', 'Barrel_Temperature_1', 'Barrel_Temperature_2', 'Barrel_Temperature_3', 'Barrel_Temperature_4', 'Barrel_Temperature_5', 'Barrel_Temperature_6', 'Barrel_Temperature_7', 'Hopper_Temperature', 'Mold_Temperature_3', 'Mold_Temperature_4']


[Historical plot output omitted; rerun this cell to display it.]

In [ ]:
num_df = df.select_dtypes(include=[np.number])

corr_mat = num_df.corr()

threshold = 0.5
mask = np.abs(corr_mat) > threshold
filtered_corr = corr_mat.where(mask).dropna(how="all").dropna(axis=1, how="all")

plt.figure(figsize=(17, 10))
sns.heatmap(filtered_corr, cmap='seismic', annot=True, fmt=".2f")
plt.title("Correlation Heatmap (|corr| > 0.5)")
plt.show()


In [ ]:
corr_unstack = corr_mat.unstack()
strong_corr = corr_unstack[(np.abs(corr_unstack) > 0.5) & (corr_unstack != 1)]
strong_corr.sort_values(ascending=False).head(20)

Filling_Time              Max_Switch_Over_Pressure    0.997808
Max_Switch_Over_Pressure  Filling_Time                0.997808
Clamp_Open_Position       Plasticizing_Position       0.996785
Plasticizing_Position     Clamp_Open_Position         0.996785
                          Max_Switch_Over_Pressure    0.996331
Max_Switch_Over_Pressure  Plasticizing_Position       0.996331
Filling_Time              Plasticizing_Position       0.995382
Plasticizing_Position     Filling_Time                0.995382
Barrel_Temperature_2      Barrel_Temperature_3        0.992912
Barrel_Temperature_3      Barrel_Temperature_2        0.992912
Injection_Time            Filling_Time                0.992421
Filling_Time              Injection_Time              0.992421
Max_Switch_Over_Pressure  Clamp_Open_Position         0.992334
Clamp_Open_Position       Max_Switch_Over_Pressure    0.992334
Injection_Time            Max_Switch_Over_Pressure    0.991439
Max_Switch_Over_Pressure  Injection_Time              0

We Use relplot to study strong correlations

After computing correlation matrix, we may find something like:

Max_Injection_Pressure strongly correlates with Switch_Over_Position

Cycle_Time correlates with Filling_Time

In [ ]:
sns.relplot(
    data=df,
    x="Max_Injection_Pressure",
    y="Switch_Over_Position",
    hue="class",
    kind="scatter",
    height=5,
    aspect=1.3
)


[Historical plot output omitted; rerun this cell to display it.]

In [ ]:
sns.relplot(
    data=df,
    x="Filling_Time",
    y="Max_Switch_Over_Pressure",
    hue="class",
    kind="scatter",
    height=5,
    aspect=1.3
)

[Historical plot output omitted; rerun this cell to display it.]

Very good for manufacturing – shows how defect shots differ from normal shots

In [ ]:
cols_to_check = [
    "Cycle_Time", "Max_Injection_Pressure",
    "Filling_Time", "Injection_Time"
]

for col in cols_to_check:
    if col in df.columns:
        plt.figure(figsize=(6,4))
        sns.boxplot(x="class", y=col, data=df)
        plt.title(f"{col} vs Pass/Fail")
        plt.tight_layout()
        plt.show()


[Historical plot output omitted; rerun this cell to display it.]

[Historical plot output omitted; rerun this cell to display it.]

[Historical plot output omitted; rerun this cell to display it.]

[Historical plot output omitted; rerun this cell to display it.]

Shows distribution difference — if Fail shifts meaningfully → very valuable.

In [ ]:
for col in cols_to_check:
    if col in df.columns:
        plt.figure(figsize=(6,4))
        sns.histplot(data=df, x=col, hue="class", kde=True, alpha=0.4)
        plt.title(f"Distribution of {col} (Pass vs Fail)")
        plt.tight_layout()
        plt.show()


[Historical plot output omitted; rerun this cell to display it.]

[Historical plot output omitted; rerun this cell to display it.]

[Historical plot output omitted; rerun this cell to display it.]

[Historical plot output omitted; rerun this cell to display it.]

Pick the top 3–5 numeric variables with highest correlation.

In [ ]:
top_cols = ["Filling_Time", "Max_Switch_Over_Pressure", "Clamp_Open_Position", "Plasticizing_Position"]
sns.pairplot(df[top_cols + ["class"]], hue="class")
plt.show()


[Historical plot output omitted; rerun this cell to display it.]

Violin plots (distribution + density) - shows manufacturing stability

In [ ]:
sns.violinplot(x="class", y="Cycle_Time", data=df)
plt.title("Cycle_Time Distribution by Pass/Fail")
plt.show()


[Historical plot output omitted; rerun this cell to display it.]

# Data Quality & Preparation

 Missing Data Analysis & MCAR/MAR/MNAR Reasoning

In [ ]:
missing = df.isna().mean().sort_values(ascending=False) * 100
display(missing.to_frame("Missing_%").head(20))

plt.figure(figsize=(10, 4))
sns.heatmap(df.isna(), cbar=False)
plt.title("Missingness Map (NaN-based)")
plt.xlabel("Columns")
plt.ylabel("Rows")
plt.show()

print("Remark: There are effectively no NaN-based missing values in the dataset.")

crosstab_reason = pd.crosstab(df["class"], df["Reason"])
display(crosstab_reason)

print('Observation: "Reason" is "None" for all Pass (Y) parts, and a specific '
      'failure reason for Fail (N) parts. This is structural missingness (not random).')

df["Reason_clean"] = df["Reason"].replace({"None": "NoDefectReason"})
print(df["Reason_clean"].value_counts())

mcar_mar_mnar = """
MCAR (Missing Completely At Random): missingness unrelated to any data.
MAR  (Missing At Random): missingness depends only on observed variables.
MNAR (Missing Not At Random): missingness depends on the missing value itself.

In this dataset, numeric columns have 0% NaNs. "Reason" is structurally
missing for good parts (Pass), not randomly. It is closer to "not applicable"
than true MCAR/MAR/MNAR. Therefore, no numeric imputation is required.
We keep all rows and simply relabel "None" as "NoDefectReason".
"""
print(mcar_mar_mnar)

                       Missing_%
Reason                 99.112056
TimeStamp               0.000000
PART_FACT_PLAN_DATE     0.000000
PART_FACT_SERIAL        0.000000
_id                     0.000000
PART_NAME               0.000000
EQUIP_CD                0.000000
EQUIP_NAME              0.000000
Injection_Time          0.000000
Filling_Time            0.000000
Plasticizing_Time       0.000000
Cycle_Time              0.000000
Clamp_Close_Time        0.000000
Cushion_Position        0.000000
Switch_Over_Position    0.000000
Plasticizing_Position   0.000000
Clamp_Open_Position     0.000000
Max_Injection_Speed     0.000000
Max_Screw_RPM           0.000000
Average_Screw_RPM       0.000000

[Historical plot output omitted; rerun this cell to display it.]

Remark: There are effectively no NaN-based missing values in the dataset.


Reason  가스  미성형  초기허용불량
class                  
0       35   16      20

Observation: "Reason" is "None" for all Pass (Y) parts, and a specific failure reason for Fail (N) parts. This is structural missingness (not random).
Reason_clean
가스        35
초기허용불량    20
미성형       16
Name: count, dtype: int64

MCAR (Missing Completely At Random): missingness unrelated to any data.
MAR  (Missing At Random): missingness depends only on observed variables.
MNAR (Missing Not At Random): missingness depends on the missing value itself.

In this dataset, numeric columns have 0% NaNs. "Reason" is structurally
missing for good parts (Pass), not randomly. It is closer to "not applicable"
than true MCAR/MAR/MNAR. Therefore, no numeric imputation is required.
We keep all rows and simply relabel "None" as "NoDefectReason".



## 3. Outlier Detection & Handling (Two Methods)

We apply **two outlier detection methods** on key process variables:
- `Injection_Time`
- `Filling_Time`
- `Average_Screw_RPM`
- `Switch_Over_Position`
- `Clamp_Open_Position`

**Methods used:**
1. **IQR method**: values outside `[Q1 − 1.5 × IQR, Q3 + 1.5 × IQR]` are considered outliers.
2. **Z-score method**: values with `|z| > 3` (three standard deviations from mean).

**Final handling decision:**
- We **keep all rows** (no row deletion), to avoid losing rare but potentially valid cycles.
- For modeling, we use **IQR-clipped versions** of these features (suffix `_clipped`).
- Z-score is used to **flag** extreme points (suffix `_z_outlier`), not to remove them.
- Original raw columns are kept for interpretation and audit.

In [ ]:
num_cols = ["Injection_Time", "Filling_Time", "Average_Screw_RPM",
            "Switch_Over_Position", "Clamp_Open_Position"]

def iqr_clip(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return series.clip(lower, upper), lower, upper

def zscore_mask(series, threshold=3.0):
    z = (series - series.mean()) / series.std()
    return (np.abs(z) > threshold)

iqr_bounds = {}
outlier_summary = []

for col in num_cols:
    # IQR clipping
    clipped, low, high = iqr_clip(df[col])
    df[col + "_clipped"] = clipped
    iqr_bounds[col] = (low, high)

    # Z-score flags (True = extreme)
    z_mask = zscore_mask(df[col])
    df[col + "_z_outlier"] = z_mask.astype(int)

    # Summary
    n_iqr = int(((df[col] < low) | (df[col] > high)).sum())
    n_z = int(z_mask.sum())
    outlier_summary.append({
        "Feature": col,
        "IQR_lower": round(low, 3),
        "IQR_upper": round(high, 3),
        "IQR_outlier_count": n_iqr,
        "Z_outlier_count_(>|3σ|)": n_z
    })

outlier_table = pd.DataFrame(outlier_summary)
display(outlier_table)

print("Remark: IQR bounds define clipping thresholds; rows outside these bounds are not dropped, "
      "but their values are clipped into the valid range. Z-score flags mark extremely rare points.")

                Feature  IQR_lower  IQR_upper  IQR_outlier_count  \
0        Injection_Time      9.415      9.695               1322   
1          Filling_Time      4.295      4.575               1322   
2     Average_Screw_RPM   -365.600    687.200                  0   
3  Switch_Over_Position      0.000      0.000                  6   
4   Clamp_Open_Position    647.990    647.990               1296   

   Z_outlier_count_(>|3σ|)  
0                        0  
1                        4  
2                        0  
3                        4  
4                        0  

Remark: IQR bounds define clipping thresholds; rows outside these bounds are not dropped, but their values are clipped into the valid range. Z-score flags mark extremely rare points.


In [ ]:
# Boxplots before vs after clipping
fig, axes = plt.subplots(2, 1, figsize=(10, 8))

sns.boxplot(data=df[num_cols], ax=axes[0], palette="Set3")
axes[0].set_title("Boxplots BEFORE Outlier Treatment")
axes[0].tick_params(axis="x", rotation=45)

sns.boxplot(data=df[[c + "_clipped" for c in num_cols]], ax=axes[1], palette="Set2")
axes[1].set_title("Boxplots AFTER IQR Clipping")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

print("Remark: After clipping, extreme tails are reduced while the bulk distribution remains similar, "
      "making models less sensitive to a few abnormal points.")

[Historical plot output omitted; rerun this cell to display it.]

Remark: After clipping, extreme tails are reduced while the bulk distribution remains similar, making models less sensitive to a few abnormal points.


In [ ]:
# Example: Distribution before/after for Injection_Time
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.histplot(df["Injection_Time"], bins=30, kde=True, ax=axes[0], color="salmon")
axes[0].set_title("Injection_Time (Raw)")

sns.histplot(df["Injection_Time_clipped"], bins=30, kde=True, ax=axes[1], color="skyblue")
axes[1].set_title("Injection_Time (IQR-clipped)")

plt.tight_layout()
plt.show()

print("Remark: Clipping slightly pulls in the extreme right tail but keeps the main mode and shape intact.")

[Historical plot output omitted; rerun this cell to display it.]

Remark: Clipping slightly pulls in the extreme right tail but keeps the main mode and shape intact.


### Outlier Handling – Final Decision

- **We do NOT drop any rows** – to avoid losing rare but possibly meaningful production conditions.
- For the selected process variables, we create **clipped versions** (e.g. `Injection_Time_clipped`) using the IQR rule.
- We also create **binary Z-score flags** (e.g. `Injection_Time_z_outlier`) to indicate extremely unusual cycles.
- **Modeling recommendation:** use `_clipped` variables as numeric inputs. The `_z_outlier` flags can be used
  as additional binary features or for further domain investigation.
- Original raw columns are kept for interpretability and traceability.

# Numerical Transformations (4+ Variables)

In [ ]:
# Log transform for skewed time feature
df["Injection_Time_log"] = np.log1p(df["Injection_Time_clipped"])

# Square-root transform for Filling_Time
df["Filling_Time_sqrt"] = np.sqrt(df["Filling_Time_clipped"])

# Z-score standardization for Max_Injection_Speed
mu_speed = df["Max_Injection_Speed"].mean()
std_speed = df["Max_Injection_Speed"].std()
df["Max_Injection_Speed_z"] = (df["Max_Injection_Speed"] - mu_speed) / std_speed

# Robust scaling for Average_Screw_RPM_clipped
median_rpm = df["Average_Screw_RPM_clipped"].median()
iqr_rpm = df["Average_Screw_RPM_clipped"].quantile(0.75) - df["Average_Screw_RPM_clipped"].quantile(0.25)
df["Average_Screw_RPM_robust"] = (df["Average_Screw_RPM_clipped"] - median_rpm) / iqr_rpm

display(df[[
    "Injection_Time_clipped", "Injection_Time_log",
    "Filling_Time_clipped", "Filling_Time_sqrt",
    "Max_Injection_Speed", "Max_Injection_Speed_z",
    "Average_Screw_RPM_clipped", "Average_Screw_RPM_robust"
]].head())

print("Remark: We applied four different numeric transformations to different variables "
      "to reduce skewness and put them on comparable scales.")

   Injection_Time_clipped  Injection_Time_log  Filling_Time_clipped  \
0                    9.59            2.359910                  4.47   
1                    9.60            2.360854                  4.48   
2                    9.60            2.360854                  4.48   
3                    9.59            2.359910                  4.48   
4                    9.59            2.359910                  4.48   

   Filling_Time_sqrt  Max_Injection_Speed  Max_Injection_Speed_z  \
0           2.114237            55.400002              -0.435508   
1           2.116601            55.299999              -0.439287   
2           2.116601            55.299999              -0.439287   
3           2.116601            55.299999              -0.439287   
4           2.116601            55.299999              -0.439287   

   Average_Screw_RPM_clipped  Average_Screw_RPM_robust  
0                      292.5                   1.00038  
1                      292.5                   1.0

Remark: We applied four different numeric transformations to different variables to reduce skewness and put them on comparable scales.


In [ ]:
# Visual check: Injection_Time before vs log-transformed
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.histplot(df["Injection_Time_clipped"], bins=30, kde=True, ax=axes[0], color="salmon")
axes[0].set_title("Injection_Time_clipped")

sns.histplot(df["Injection_Time_log"], bins=30, kde=True, ax=axes[1], color="skyblue")
axes[1].set_title("Injection_Time_log")

plt.tight_layout()
plt.show()

print("Remark: Log transformation makes the distribution more symmetric and less heavy-tailed.")

[Historical plot output omitted; rerun this cell to display it.]

Remark: Log transformation makes the distribution more symmetric and less heavy-tailed.


# Categorical Encoding (4 Variables)

In [ ]:

cat_cols = ["PART_NAME", "EQUIP_CD", "EQUIP_NAME", "Reason"]
print("Categorical variables to encode:", cat_cols)

# One-hot encode these 4 variables
df_enc = pd.get_dummies(df, columns=cat_cols, drop_first=False)

# --------------------------------------------------------------
# Show tables for each encoded categorical variable
# --------------------------------------------------------------

for col in cat_cols:
    dummy_cols = [c for c in df_enc.columns if c.startswith(col + "_")]

    if not dummy_cols:
        print(f"\n[{col}] – no dummy columns created (maybe only one category).")
        continue

    print(f"\n[{col}] – original + one-hot encoded columns (first 10 rows)")
    preview = pd.concat(
        [
            df[[col]].head(10),
            df_enc[dummy_cols].head(10),
        ],
        axis=1,
    )
    display(preview)

    print(
        f"Total unique categories in '{col}': {df[col].nunique()} "
        f"→ dummy columns created: {len(dummy_cols)}"
    )

print(
    "\nRemark: We used one-hot encoding for nominal categorical variables so that "
    "the model can learn a separate effect for each category without imposing a "
    "fake numeric order."
)

# --------------------------------------------------------------
# Frequency encoding for EQUIP_CD (optional but nice analysis)
# --------------------------------------------------------------

if "EQUIP_CD" in df.columns:
    freq_map = df["EQUIP_CD"].value_counts(normalize=True)
    df_enc["EQUIP_CD_freq"] = df["EQUIP_CD"].map(freq_map)

    print("\n[EQUIP_CD] – frequency encoding table (top 10 codes)")
    equip_freq_table = (
        df[["EQUIP_CD"]]
        .assign(EQUIP_CD_freq=df["EQUIP_CD"].map(freq_map))
        .drop_duplicates()
        .sort_values("EQUIP_CD_freq", ascending=False)
        .head(10)
    )
    display(equip_freq_table)

    print(
        "Remark: EQUIP_CD_freq represents how common each machine code is (0–1). "
        "This keeps the feature low-dimensional while still capturing usage frequency."
    )
else:
    print("\nColumn 'EQUIP_CD' not found – skipping frequency encoding.")


Categorical variables to encode: ['PART_NAME', 'EQUIP_CD', 'EQUIP_NAME', 'Reason']

[PART_NAME] – original + one-hot encoded columns (first 10 rows)


               PART_NAME  PART_NAME_CN7 W/S SIDE MLD'G LH  \
0  CN7 W/S SIDE MLD'G RH                            False   
1  CN7 W/S SIDE MLD'G RH                            False   
2  CN7 W/S SIDE MLD'G LH                             True   
3  CN7 W/S SIDE MLD'G LH                             True   
4  CN7 W/S SIDE MLD'G RH                            False   
5  CN7 W/S SIDE MLD'G RH                            False   
6  CN7 W/S SIDE MLD'G LH                             True   
7  CN7 W/S SIDE MLD'G RH                            False   
8  CN7 W/S SIDE MLD'G LH                             True   
9  CN7 W/S SIDE MLD'G RH                            False   

   PART_NAME_CN7 W/S SIDE MLD'G RH  PART_NAME_JX1 W/S SIDE MLD'G RH  \
0                             True                            False   
1                             True                            False   
2                            False                            False   
3                            False          

Total unique categories in 'PART_NAME': 6 → dummy columns created: 6

[EQUIP_CD] – original + one-hot encoded columns (first 10 rows)


  EQUIP_CD  EQUIP_CD_S01  EQUIP_CD_S12  EQUIP_CD_S14
0      S14         False         False          True
1      S14         False         False          True
2      S14         False         False          True
3      S14         False         False          True
4      S14         False         False          True
5      S14         False         False          True
6      S14         False         False          True
7      S14         False         False          True
8      S14         False         False          True
9      S14         False         False          True

Total unique categories in 'EQUIP_CD': 3 → dummy columns created: 3

[EQUIP_NAME] – original + one-hot encoded columns (first 10 rows)


   EQUIP_NAME  EQUIP_NAME_1800TON-우진  EQUIP_NAME_650톤-우진  \
0  650톤-우진2호기                  False               False   
1  650톤-우진2호기                  False               False   
2  650톤-우진2호기                  False               False   
3  650톤-우진2호기                  False               False   
4  650톤-우진2호기                  False               False   
5  650톤-우진2호기                  False               False   
6  650톤-우진2호기                  False               False   
7  650톤-우진2호기                  False               False   
8  650톤-우진2호기                  False               False   
9  650톤-우진2호기                  False               False   

   EQUIP_NAME_650톤-우진2호기  
0                   True  
1                   True  
2                   True  
3                   True  
4                   True  
5                   True  
6                   True  
7                   True  
8                   True  
9                   True  

Total unique categories in 'EQUIP_NAME': 3 → dummy columns created: 3

[Reason] – original + one-hot encoded columns (first 10 rows)


  Reason Reason_clean  Reason_가스  Reason_미성형  Reason_초기허용불량
0    NaN          NaN      False       False          False
1    NaN          NaN      False       False          False
2    NaN          NaN      False       False          False
3    NaN          NaN      False       False          False
4    NaN          NaN      False       False          False
5    NaN          NaN      False       False          False
6    NaN          NaN      False       False          False
7    NaN          NaN      False       False          False
8    NaN          NaN      False       False          False
9    NaN          NaN      False       False          False

Total unique categories in 'Reason': 3 → dummy columns created: 4

Remark: We used one-hot encoding for nominal categorical variables so that the model can learn a separate effect for each category without imposing a fake numeric order.

[EQUIP_CD] – frequency encoding table (top 10 codes)


     EQUIP_CD  EQUIP_CD_freq
0         S14        0.99950
3021      S01        0.00025
4421      S12        0.00025

Remark: EQUIP_CD_freq represents how common each machine code is (0–1). This keeps the feature low-dimensional while still capturing usage frequency.


In [ ]:
df

                           _id            TimeStamp     PART_FACT_PLAN_DATE  \
0     5f8928bb9c0189cc666ef19b  2020-10-16 04:57:47  2020-10-16 오전 12:00:00   
1     5f8928de9c0189cc666ef20b  2020-10-16 04:58:48  2020-10-16 오전 12:00:00   
2     5f8928df9c0189cc666ef213  2020-10-16 04:58:48  2020-10-16 오전 12:00:00   
3     5f8928f39c0189cc666ef25e  2020-10-16 04:59:48  2020-10-16 오전 12:00:00   
4     5f8928f59c0189cc666ef265  2020-10-16 04:59:48  2020-10-16 오전 12:00:00   
...                        ...                  ...                     ...   
7991  5fa112859c0189cc66dabe34  2020-11-03 08:17:36  2020-11-03 오전 12:00:00   
7992  5fa112869c0189cc66dabe42  2020-11-03 08:18:37  2020-11-03 오전 12:00:00   
7993  5fa112879c0189cc66dabe50  2020-11-03 08:18:37  2020-11-03 오전 12:00:00   
7994  5fa112bb9c0189cc66dac22a  2020-11-03 08:19:35  2020-11-03 오전 12:00:00   
7995  5fa112bc9c0189cc66dac23a  2020-11-03 08:19:35  2020-11-03 오전 12:00:00   

      PART_FACT_SERIAL              PART_NAME EQUIP

# Modeling

In [ ]:
!pip install imbalanced-learn

Pipelines + SMOTE + K-fold CV

In [ ]:
import numpy as np
import pandas as pd

# df_enc should be your encoded dataframe with 'class' as target
assert "class" in df_enc.columns, "Target column 'class' not found in df_enc."

# Separate features and target
X_full = df_enc.drop(columns=["class"])
y = df_enc["class"]

# Keep only numeric columns for modeling
numeric_cols = X_full.select_dtypes(include=[np.number]).columns.tolist()
non_numeric_cols = [c for c in X_full.columns if c not in numeric_cols]

print("Dropping non-numeric columns (cannot be scaled):")
print(non_numeric_cols)

X = X_full[numeric_cols]

print("Final X shape:", X.shape)
print("y distribution:")
display(y.value_counts().to_frame("count"))
display(y.value_counts(normalize=True).to_frame("percentage"))


Dropping non-numeric columns (cannot be scaled):
['_id', 'TimeStamp', 'PART_FACT_PLAN_DATE', 'Reason_clean', "PART_NAME_CN7 W/S SIDE MLD'G LH", "PART_NAME_CN7 W/S SIDE MLD'G RH", "PART_NAME_JX1 W/S SIDE MLD'G RH", "PART_NAME_RG3 MOLD'G W/SHLD, LH", "PART_NAME_RG3 MOLD'G W/SHLD, RH", 'PART_NAME_SP2 CVR ROOF RACK CTR, RH', 'EQUIP_CD_S01', 'EQUIP_CD_S12', 'EQUIP_CD_S14', 'EQUIP_NAME_1800TON-우진', 'EQUIP_NAME_650톤-우진', 'EQUIP_NAME_650톤-우진2호기', 'Reason_가스', 'Reason_미성형', 'Reason_초기허용불량']
Final X shape: (7996, 52)
y distribution:


       count
class       
1       7925
0         71

       percentage
class            
1        0.991121
0        0.008879

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTrain shape:", X_train.shape, " Test shape:", X_test.shape)



Train shape: (6396, 52)  Test shape: (1600, 52)


In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, PowerTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix, make_scorer
)
from sklearn.model_selection import StratifiedKFold, cross_validate

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline  # NOTE: from imblearn, not sklearn

def make_log_reg():
    return LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        solver="saga",
        n_jobs=-1
    )

pipelines = {
    "No Scaling (raw + SMOTE)": Pipeline(steps=[
        ("smote", SMOTE(random_state=42)),
        ("clf", make_log_reg())
    ]),
    "StandardScaler + SMOTE": Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("smote", SMOTE(random_state=42)),
        ("clf", make_log_reg())
    ]),
    "MinMaxScaler + SMOTE": Pipeline(steps=[
        ("scaler", MinMaxScaler()),
        ("smote", SMOTE(random_state=42)),
        ("clf", make_log_reg())
    ]),
    "YeoJohnson + StandardScaler + SMOTE": Pipeline(steps=[
        ("power", PowerTransformer(method="yeo-johnson", standardize=False)),
        ("scaler", StandardScaler()),
        ("smote", SMOTE(random_state=42)),
        ("clf", make_log_reg())
    ]),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
    "roc_auc": "roc_auc"
}

cv_results = {}

for name, pipe in pipelines.items():
    print("\n" + "="*80)
    print(f"Pipeline: {name}")
    print("="*80)

    scores = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=False,
        n_jobs=-1
    )

    cv_results[name] = scores

    summary = {
        metric: f"{scores['test_' + metric].mean():.4f} ± {scores['test_' + metric].std():.4f}"
        for metric in scoring.keys()
    }
    display(pd.DataFrame(summary, index=[name]))



Pipeline: No Scaling (raw + SMOTE)


                                 accuracy        precision           recall  \
No Scaling (raw + SMOTE)  0.8160 ± 0.0117  0.9985 ± 0.0018  0.8156 ± 0.0120   

                                       f1          roc_auc  
No Scaling (raw + SMOTE)  0.8978 ± 0.0073  0.9082 ± 0.0271  


Pipeline: StandardScaler + SMOTE


                               accuracy        precision           recall  \
StandardScaler + SMOTE  0.8318 ± 0.0049  0.9989 ± 0.0009  0.8312 ± 0.0044   

                                     f1          roc_auc  
StandardScaler + SMOTE  0.9073 ± 0.0029  0.9168 ± 0.0243  


Pipeline: MinMaxScaler + SMOTE


                             accuracy        precision           recall  \
MinMaxScaler + SMOTE  0.8005 ± 0.0140  0.9986 ± 0.0018  0.7998 ± 0.0147   

                                   f1          roc_auc  
MinMaxScaler + SMOTE  0.8882 ± 0.0088  0.8947 ± 0.0345  


Pipeline: YeoJohnson + StandardScaler + SMOTE


                                            accuracy        precision  \
YeoJohnson + StandardScaler + SMOTE  0.8382 ± 0.0076  0.9987 ± 0.0010   

                                              recall               f1  \
YeoJohnson + StandardScaler + SMOTE  0.8378 ± 0.0069  0.9112 ± 0.0045   

                                             roc_auc  
YeoJohnson + StandardScaler + SMOTE  0.9324 ± 0.0182  

# Comparison

In [ ]:
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler,
    PowerTransformer, QuantileTransformer
)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    make_scorer, precision_score, recall_score, f1_score
)
from imblearn.pipeline import Pipeline   # IMPORTANT: imblearn Pipeline
from imblearn.over_sampling import SMOTE

# Logistic Regression factory
def make_log_reg():
    return LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        solver="saga",
        n_jobs=-1
    )

pipelines = {
    "No Scaling": Pipeline([
        ("smote", SMOTE(random_state=42)),
        ("clf", make_log_reg())
    ]),

    "StandardScaler": Pipeline([
        ("scaler", StandardScaler()),
        ("smote", SMOTE(random_state=42)),
        ("clf", make_log_reg())
    ]),

    "MinMaxScaler": Pipeline([
        ("scaler", MinMaxScaler()),
        ("smote", SMOTE(random_state=42)),
        ("clf", make_log_reg())
    ]),

    "RobustScaler": Pipeline([
        ("scaler", RobustScaler()),
        ("smote", SMOTE(random_state=42)),
        ("clf", make_log_reg())
    ]),

    "YeoJohnson + StandardScaler": Pipeline([
        ("power", PowerTransformer(method="yeo-johnson", standardize=False)),
        ("scaler", StandardScaler()),
        ("smote", SMOTE(random_state=42)),
        ("clf", make_log_reg())
    ]),

    "QuantileGaussian": Pipeline([
        ("qt", QuantileTransformer(output_distribution="normal")),
        ("smote", SMOTE(random_state=42)),
        ("clf", make_log_reg())
    ]),

    "QuantileUniform": Pipeline([
        ("qt", QuantileTransformer(output_distribution="uniform")),
        ("smote", SMOTE(random_state=42)),
        ("clf", make_log_reg())
    ])
}


In [ ]:
# Use stratified K-fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Multiple metrics
scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
    "roc_auc": "roc_auc"
}


In [ ]:
import pandas as pd

cv_summary = []

for name, pipe in pipelines.items():
    print(f"\n🔄 Running pipeline: {name}")

    scores = cross_validate(
        pipe, X_train, y_train,
        scoring=scoring,
        cv=cv,
        n_jobs=-1,
        return_train_score=False
    )

    cv_summary.append({
        "Pipeline": name,
        "Accuracy": scores["test_accuracy"].mean(),
        "Precision": scores["test_precision"].mean(),
        "Recall": scores["test_recall"].mean(),
        "F1-score": scores["test_f1"].mean(),
        "ROC-AUC": scores["test_roc_auc"].mean()
    })

cv_results_df = pd.DataFrame(cv_summary)
cv_results_df = cv_results_df.sort_values(by="F1-score", ascending=False)

print("\n📊 Cross-Validation Summary:")
display(cv_results_df)



🔄 Running pipeline: No Scaling

🔄 Running pipeline: StandardScaler

🔄 Running pipeline: MinMaxScaler

🔄 Running pipeline: RobustScaler

🔄 Running pipeline: YeoJohnson + StandardScaler

🔄 Running pipeline: QuantileGaussian

🔄 Running pipeline: QuantileUniform

📊 Cross-Validation Summary:


                      Pipeline  Accuracy  Precision    Recall  F1-score  \
5             QuantileGaussian  0.888209   0.997735  0.889255  0.940272   
6              QuantileUniform  0.843964   0.998519  0.843822  0.914638   
4  YeoJohnson + StandardScaler  0.838180   0.998677  0.837828  0.911196   
1               StandardScaler  0.831770   0.998860  0.831202  0.907346   
0                   No Scaling  0.815979   0.998461  0.815583  0.897753   
2                 MinMaxScaler  0.800499   0.998637  0.799808  0.888151   
3                 RobustScaler  0.487664   0.997058  0.484295  0.634764   

    ROC-AUC  
5  0.931955  
6  0.925682  
4  0.932435  
1  0.916873  
0  0.908205  
2  0.894657  
3  0.800244  

In [ ]:
best_name = cv_results_df.iloc[0]["Pipeline"]
print(f"\n🏆 Best Pipeline based on F1-score: {best_name}")

best_pipeline = pipelines[best_name]
best_pipeline.fit(X_train, y_train)

y_pred = best_pipeline.predict(X_test)



🏆 Best Pipeline based on F1-score: QuantileGaussian


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    df_cm = pd.DataFrame(cm,
                         index=["Actual 0", "Actual 1"],
                         columns=["Pred 0", "Pred 1"])
    print(df_cm)

    plt.figure(figsize=(5,4))
    sns.heatmap(df_cm, annot=True, fmt="d", cmap="Blues")
    plt.title(title)
    plt.show()


In [ ]:
plot_confusion(y_test, y_pred, f"Best Pipeline: {best_name}")


          Pred 0  Pred 1
Actual 0       9       5
Actual 1     174    1412


[Historical plot output omitted; rerun this cell to display it.]

# Plot Performance Comparison of Pipelines

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# For pretty plots
plt.style.use("seaborn-v0_8")
sns.set_theme(font_scale=1.2)

metrics = ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]

fig, axes = plt.subplots(3, 2, figsize=(16, 18))
axes = axes.flatten()

for i, metric in enumerate(metrics):
    sns.barplot(
        data=cv_results_df,
        x=metric,
        y="Pipeline",
        ax=axes[i],
        palette="Blues_r"
    )
    axes[i].set_title(f"{metric} Comparison Across Pipelines")
    axes[i].set_xlabel(metric)
    axes[i].set_ylabel("Pipeline")
    axes[i].grid(True, axis='x', linestyle='--', alpha=0.5)

# Hide last empty subplot if 3x2 = 6 axes but we use 5
axes[-1].set_visible(False)

plt.tight_layout()
plt.show()


/tmp/ipython-input-2541132150.py:14: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(
/tmp/ipython-input-2541132150.py:14: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(
/tmp/ipython-input-2541132150.py:14: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(
/tmp/ipython-input-2541132150.py:14: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(
/tmp/ipython-input-2541132150.py:14: FutureWarning: 

Passin

[Historical chart exported to results/figures/preprocessing-comparison.png]

Performance Analysis of Multiple Preprocessing Pipelines

We evaluated seven pipelines combining different feature preprocessing methods (StandardScaler, MinMaxScaler, RobustScaler, PowerTransform, QuantileTransform) together with SMOTE and Logistic Regression (balanced). Performance was measured using 5-fold Stratified Cross-Validation.

The five metrics compared were:

Accuracy — overall correctness

Precision — correctness of predicted defects

Recall — how many defects were actually detected

F1-score — balance of precision & recall

ROC-AUC — ranking ability of the model

Why this pipeline is best

The QuantileGaussian transform maps feature distributions into a Gaussian (normal) distribution, which logistic regression works extremely well with.

It also reduces the impact of outliers and makes skewed variables more regular.

The model achieves excellent recall (0.889), meaning it captures a large portion of actual defects.

It has the strongest F1-score, showing the best trade-off between precision and recall.

Highest or near-highest ROC-AUC, meaning best ranking ability.

Conclusion:
▶️ This pipeline achieves the best overall predictive performance and is the recommended preprocessing method.

# Compare Multiple Classifiers Under the SAME Best Preprocessing Pipeline

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

# Optional: XGBoost
# !pip install xgboost
from xgboost import XGBClassifier

def make_log_reg():
    return LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        solver="saga",
        n_jobs=-1
    )

classifiers = {
    "Logistic Regression": make_log_reg(),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced",
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingClassifier(),

    "Linear SVM": LinearSVC(
        class_weight="balanced",
        max_iter=5000
    ),

    "KNN": KNeighborsClassifier(n_neighbors=5),

    "Naive Bayes": GaussianNB(),

    "XGBoost": XGBClassifier(
        eval_metric="logloss",
        scale_pos_weight=10,
        learning_rate=0.1,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )
}



In [ ]:
from sklearn.preprocessing import QuantileTransformer
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

def make_pipeline(clf):
    return Pipeline([
        ("qt", QuantileTransformer(output_distribution="normal")),
        ("smote", SMOTE(random_state=42, sampling_strategy=0.3)),
        ("clf", clf)
    ])


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
    "roc_auc": "roc_auc"
}

clf_results = []

for name, clf in classifiers.items():
    print(f"\n🔄 Running classifier: {name}")
    pipe = make_pipeline(clf)

    scores = cross_validate(
        pipe, X_train, y_train,
        scoring=scoring,
        cv=cv,
        n_jobs=-1,
        return_train_score=False
    )

    clf_results.append({
        "Classifier": name,
        "Accuracy": scores["test_accuracy"].mean(),
        "Precision": scores["test_precision"].mean(),
        "Recall": scores["test_recall"].mean(),
        "F1-score": scores["test_f1"].mean(),
        "ROC-AUC": scores["test_roc_auc"].mean()
    })

clf_df = pd.DataFrame(clf_results)
clf_df = clf_df.sort_values(by="F1-score", ascending=False)

display(clf_df)



🔄 Running classifier: Logistic Regression

🔄 Running classifier: Random Forest

🔄 Running classifier: Decision Tree

🔄 Running classifier: Gradient Boosting

🔄 Running classifier: Linear SVM

🔄 Running classifier: KNN

🔄 Running classifier: Naive Bayes

🔄 Running classifier: XGBoost


            Classifier  Accuracy  Precision    Recall  F1-score   ROC-AUC
7              XGBoost  0.992495   0.993722  0.998738  0.996223  0.948540
3    Gradient Boosting  0.992026   0.994965  0.997003  0.995981  0.937134
1        Random Forest  0.991245   0.994025  0.997161  0.995590  0.950285
2        Decision Tree  0.986554   0.994310  0.992112  0.993209  0.680905
5                  KNN  0.960600   0.996091  0.964032  0.979796  0.822557
0  Logistic Regression  0.887271   0.998075  0.887993  0.939728  0.931159
4           Linear SVM  0.877108   0.998058  0.877738  0.933897  0.929935
6          Naive Bayes  0.174031   0.999787  0.166722  0.206302  0.908188

In [ ]:
plt.figure(figsize=(12,6))
sns.barplot(data=clf_df, x="F1-score", y="Classifier", palette="Blues_r")
plt.title("Classifier Comparison (F1-score)")
plt.show()


/tmp/ipython-input-1594686793.py:2: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=clf_df, x="F1-score", y="Classifier", palette="Blues_r")


[Historical chart exported to results/figures/classifier-comparison.png]

In [ ]:
best_clf_name = clf_df.iloc[0]["Classifier"]
print("🏆 Best classifier:", best_clf_name)

best_pipeline = make_pipeline(classifiers[best_clf_name])
best_pipeline.fit(X_train, y_train)

y_pred = best_pipeline.predict(X_test)


🏆 Best classifier: XGBoost


In [ ]:
plot_confusion(y_test, y_pred, f"Best Classifier: {best_clf_name}")

          Pred 0  Pred 1
Actual 0       6       8
Actual 1       2    1584


[Historical chart exported to results/figures/test-confusion-matrix.png]